# Phase 1 — Generate Probe Dataset

This notebook collects the labeled dataset needed to train and evaluate the SE probes.

**What it produces**: `backend/data/probe_dataset_llama3-8b_triviaqa.pkl`

**Per-question pipeline**:
1. `engine.generate_responses(question, num_samples=5)` — multi-sample generation with top-2 logit capture
2. `engine.find_semantic_clusters(question, answers)` — LLM-based semantic clustering
3. Compute **Energy teacher** (`energy_score_raw`) from `cal_flow → sum_normalize`
4. Compute **Entropy teacher** (`entropy_score_raw`) from `cluster_assignment_entropy`
5. Compute **correctness** via normalized exact match (TriviaQA)
6. Run a **separate forward pass** on (prompt + answer) to extract hidden states
7. Extract TBG hidden state `(33, 4096)` and SLT hidden state `(33, 4096)`
8. Extract logit summary features from `generated_data[0]`

> **Note on hidden state extraction**: We use a SEPARATE `model(input_ids, output_hidden_states=True)` call
> AFTER generation completes. We do NOT use `model.generate(output_hidden_states=True)` because that
> stores 512 steps × 33 layers × 4096 dims ≈ 1.3 GB per generation call — completely infeasible.

## Section 1 — Setup and Imports

In [1]:
import sys
import os
import re
import math
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Add backend to path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BACKEND_PATH = os.path.join(REPO_ROOT, 'backend')
DATA_DIR = os.path.join(REPO_ROOT, 'backend', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Repo root : D:\Github Repositories\SemanticEnergy
Data dir  : D:\Github Repositories\SemanticEnergy\backend\data
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM total: 12.0 GB


## Section 2 — Dataset Loading

In [2]:
from datasets import load_dataset

# Load TriviaQA validation split
# We use the 'rc' config which has clean reference answers
print("Loading TriviaQA validation split...")
triviaqa = load_dataset('trivia_qa', 'rc', split='validation', trust_remote_code=True)

print(f"Dataset size: {len(triviaqa)} examples")
print(f"Features: {list(triviaqa.features.keys())}")
print()
# Inspect one example
ex = triviaqa[0]
print("Example question:", ex['question'])
print("Answer value:    ", ex['answer']['value'])
print("Answer aliases:  ", ex['answer']['aliases'][:5])

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# Configuration
NUM_QUESTIONS = 1000     # total questions to process (adjust down for faster test runs)
NUM_SAMPLES   = 5        # diverse responses per question
CHECKPOINT_EVERY = 100   # save intermediate checkpoint every N questions
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'

# Subset dataset
dataset_subset = triviaqa.select(range(NUM_QUESTIONS))
print(f"Processing {NUM_QUESTIONS} questions from TriviaQA validation split")
print(f"Samples per question: {NUM_SAMPLES}")
print(f"Model: {MODEL_ID}")

## Section 3 — Model Loading

In [ ]:
from engine import SemanticEngine, cal_flow, sum_normalize

print(f"Loading {MODEL_ID}...")
engine = SemanticEngine(model_id=MODEL_ID, use_8bit=True)
print("Model ready.")

## Section 4 — Helper Functions

In [ ]:
# ── Correctness: Normalized Substring Match (TriviaQA, generative setting) ──────

def normalize_answer(text):
    """Normalize: lowercase, remove articles, strip punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return ' '.join(text.split())

def is_correct_triviaqa(predicted_answer, reference_aliases):
    """
    Return 1.0 if any reference alias appears as a substring in the predicted answer
    after normalization. This is the standard evaluation for generative LLMs on TriviaQA:
    the model generates a full sentence ('The capital of France is Paris.') and we check
    whether the correct answer ('Paris') is contained within it.
    
    Note: exact match would wrongly mark 'The capital of France is Paris.' as incorrect.
    """
    norm_pred = normalize_answer(predicted_answer)
    for ref in reference_aliases:
        if normalize_answer(ref) in norm_pred:
            return 1.0
    return 0.0

# Sanity checks
assert is_correct_triviaqa("The capital of France is Paris.", ["Paris"]) == 1.0  # full sentence
assert is_correct_triviaqa("The Beatles", ["The Beatles", "Beatles"]) == 1.0
assert is_correct_triviaqa("the beatles", ["The Beatles"]) == 1.0
assert is_correct_triviaqa("Rolling Stones", ["The Beatles", "Beatles"]) == 0.0
assert is_correct_triviaqa("paris is the capital", ["Paris"]) == 1.0
print("Correctness function: PASS")

In [ ]:
# ── Entropy Teacher ──────────────────────────────────────────────────────────────

def cluster_assignment_entropy(clusters):
    """Shannon entropy over cluster size distribution. Exact formula from SEP."""
    sizes = [len(c) for c in clusters]
    n = sum(sizes)
    probs = [s / n for s in sizes]
    return -sum(p * math.log(p + 1e-10) for p in probs)

# Sanity check
assert cluster_assignment_entropy([[0, 1, 2, 3, 4]]) < 0.01  # near-zero: all same
assert cluster_assignment_entropy([[0],[1],[2],[3],[4]]) > 1.5  # high: all different
print("Entropy function: PASS")

In [ ]:
# ── Hidden State Extraction ───────────────────────────────────────────────────────

def extract_hidden_states(engine, prompt_text, answer_text):
    """
    Run a SEPARATE forward pass on (prompt + answer) to extract hidden states.
    
    Returns:
        tbg_hidden: np.ndarray (num_layers, hidden_dim) — last token of prompt (TBG)
        slt_hidden: np.ndarray (num_layers, hidden_dim) — second-to-last generated token (SLT)
    
    We do NOT use model.generate(output_hidden_states=True) because that stores
    512 steps × 33 layers × 4096 dims ≈ 1.3 GB per call.
    """
    tokenizer = engine.tokenizer
    model = engine.model
    
    # Build prompt text (same format as generate_responses)
    messages = [{'role': 'user', 'content': prompt_text}]
    prompt_only = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # Tokenize prompt separately to know where prompt ends
    prompt_ids = tokenizer(prompt_only, return_tensors='pt').input_ids
    prompt_len = prompt_ids.shape[1]
    
    # Tokenize full sequence (prompt + answer)
    full_text = prompt_only + answer_text
    full_inputs = tokenizer(full_text, return_tensors='pt').to('cuda:0')
    full_len = full_inputs.input_ids.shape[1]
    
    # Need at least 2 generated tokens for SLT
    if full_len <= prompt_len + 1:
        return None, None
    
    with torch.no_grad():
        outputs = model(
            **full_inputs,
            output_hidden_states=True
        )
    
    # hidden_states: tuple of (batch, seq_len, hidden_dim) — one per layer including embedding
    # Stack to (num_layers, seq_len, hidden_dim)
    hidden = torch.stack(outputs.hidden_states, dim=0)  # (num_layers, 1, seq_len, hidden_dim)
    hidden = hidden[:, 0, :, :].float().cpu()             # (num_layers, seq_len, hidden_dim)
    
    # TBG: last token of prompt
    tbg_idx = prompt_len - 1
    tbg_hidden = hidden[:, tbg_idx, :].numpy()   # (num_layers, hidden_dim)
    
    # SLT: second-to-last token of the full sequence (i.e., token before EOS/last)
    slt_idx = full_len - 2
    slt_hidden = hidden[:, slt_idx, :].numpy()   # (num_layers, hidden_dim)
    
    del outputs, hidden
    torch.cuda.empty_cache()
    
    return tbg_hidden, slt_hidden

print("extract_hidden_states defined.")

In [ ]:
# ── Logit Feature Extraction ─────────────────────────────────────────────────────

def extract_logit_feats(generated_data_0):
    """
    Extract logit-centered features from the first generated sample.

    Primary features (logit-centered):
      mean_chosen_logit   — mean logit of chosen token across steps
      min_chosen_logit    — min logit of chosen token (weakest step)
      std_chosen_logit    — std of chosen token logit
      mean_logit_margin   — mean(top1 - top2), inf margins excluded
      min_logit_margin    — min finite margin
      std_logit_margin    — std of finite margins
      answer_len          — number of generated tokens

    Optional ablation:
      mean_chosen_prob    — mean softmax prob of chosen token
      min_chosen_prob     — min softmax prob

    Note: some steps produce top2 = -inf (model is fully certain, all other tokens
    suppressed). Those margins (top1 - (-inf) = +inf) are excluded from statistics
    to avoid inf/nan propagating into probe training.
    """
    logits = np.array(generated_data_0['logits'])
    probs  = np.array(generated_data_0['probs'])
    answer_len = len(logits)

    if answer_len == 0:
        return {
            'mean_chosen_logit': 0.0, 'min_chosen_logit': 0.0, 'std_chosen_logit': 0.0,
            'mean_logit_margin': 0.0, 'min_logit_margin': 0.0, 'std_logit_margin': 0.0,
            'answer_len': 0,
            'mean_chosen_prob': 0.0, 'min_chosen_prob': 0.0,
        }

    mean_logit = float(np.mean(logits))
    min_logit  = float(np.min(logits))
    std_logit  = float(np.std(logits))

    top2_logits = generated_data_0.get('top2_logits', None)
    if top2_logits is not None and len(top2_logits) > 0:
        raw_margins = np.array([t1 - t2 for t1, t2 in top2_logits])
        # Filter out inf margins (top2 = -inf means model was fully certain at that step)
        finite_margins = raw_margins[np.isfinite(raw_margins)]
        if len(finite_margins) > 0:
            mean_margin = float(np.mean(finite_margins))
            min_margin  = float(np.min(finite_margins))
            std_margin  = float(np.std(finite_margins)) if len(finite_margins) > 1 else 0.0
        else:
            # All steps were fully certain — use 0 as a sentinel for "no uncertainty"
            mean_margin = min_margin = std_margin = 0.0
    else:
        mean_margin = min_margin = std_margin = 0.0

    mean_prob = float(np.mean(probs))
    min_prob  = float(np.min(probs))

    return {
        'mean_chosen_logit':  mean_logit,
        'min_chosen_logit':   min_logit,
        'std_chosen_logit':   std_logit,
        'mean_logit_margin':  mean_margin,
        'min_logit_margin':   min_margin,
        'std_logit_margin':   std_margin,
        'answer_len':         answer_len,
        'mean_chosen_prob':   mean_prob,
        'min_chosen_prob':    min_prob,
    }

print("extract_logit_feats defined.")

## Section 5 — Single Record Generation (Test)

Run one example end-to-end before the full loop to catch any issues.

In [ ]:
# Test run on a single example
test_ex = dataset_subset[0]
test_q  = test_ex['question']
test_refs = test_ex['answer']['aliases']

print(f"Question: {test_q}")
print(f"References: {test_refs[:3]}")
print()

# Step 1: Generate responses
print("[1/4] Generating responses...")
gen_data = engine.generate_responses(test_q, num_samples=NUM_SAMPLES)
print(f"Generated {len(gen_data)} responses")
print(f"Main answer: {gen_data[0]['answer'][:100]}")
print(f"Logit feats keys: {list(gen_data[0].keys())}")

# Step 2: Find clusters
print("\n[2/4] Finding semantic clusters...")
answer_texts = [d['answer'] for d in gen_data]
clusters = engine.find_semantic_clusters(test_q, answer_texts)
print(f"Clusters: {clusters}")

# Step 3: Compute teachers
print("\n[3/4] Computing teacher signals...")
probs_list  = [d['probs'] for d in gen_data]
logits_list = [d['logits'] for d in gen_data]
probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)
cluster_energies = sum_normalize(logits_se)
main_cluster_idx = next(idx for idx, c in enumerate(clusters) if 0 in c)
energy_score_raw = cluster_energies[main_cluster_idx]
entropy_score_raw = cluster_assignment_entropy(clusters)
print(f"energy_score_raw  = {energy_score_raw:.4f}")
print(f"entropy_score_raw = {entropy_score_raw:.4f}")

# Step 4: Correctness
correctness = is_correct_triviaqa(gen_data[0]['answer'], test_refs)
print(f"correctness       = {correctness}")

In [ ]:
# Step 5: Extract hidden states
print("[4/4] Extracting hidden states...")
tbg_hidden, slt_hidden = extract_hidden_states(engine, test_q, gen_data[0]['answer'])

if tbg_hidden is not None:
    print(f"TBG hidden shape: {tbg_hidden.shape}  (expected: (num_layers, 4096))")
    print(f"SLT hidden shape: {slt_hidden.shape}")
    print(f"Memory per record: {tbg_hidden.nbytes + slt_hidden.nbytes} bytes")
else:
    print("WARNING: hidden state extraction returned None — answer too short?")

# Logit features
logit_feats = extract_logit_feats(gen_data[0])
print(f"\nLogit features:")
for k, v in logit_feats.items():
    print(f"  {k}: {v}")

## Section 6 — Full Generation Loop

In [ ]:
def generate_record(engine, example, num_samples=5):
    """
    Generate a full record for one TriviaQA example.
    Returns a record dict or None on error.
    """
    question = example['question']
    uid      = example['question_id']
    refs     = example['answer']['aliases']
    
    # 1. Generate
    gen_data = engine.generate_responses(question, num_samples=num_samples)
    main_answer = gen_data[0]['answer']
    answer_texts = [d['answer'] for d in gen_data]
    
    # 2. Cluster
    clusters = engine.find_semantic_clusters(question, answer_texts)
    main_cluster_idx = next(idx for idx, c in enumerate(clusters) if 0 in c)
    
    # 3. Energy teacher
    probs_list  = [d['probs']  for d in gen_data]
    logits_list = [d['logits'] for d in gen_data]
    probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)
    cluster_energies = sum_normalize(logits_se)
    energy_score_raw = cluster_energies[main_cluster_idx]
    
    # 4. Entropy teacher (same clusters)
    entropy_score_raw = cluster_assignment_entropy(clusters)
    
    # 5. Correctness (normalized EM)
    correctness = is_correct_triviaqa(main_answer, refs)
    
    # 6. Hidden states (separate forward pass)
    tbg_hidden, slt_hidden = extract_hidden_states(engine, question, main_answer)
    
    # 7. Logit features from sample 0
    logit_feats = extract_logit_feats(gen_data[0])
    
    return {
        'uid':                     uid,
        'question':                question,
        'main_answer':             main_answer,
        'correctness':             correctness,
        'energy_score_raw':        energy_score_raw,
        'entropy_score_raw':       entropy_score_raw,
        'energy_label':            None,   # set in notebook 02
        'entropy_label':           None,   # set in notebook 02
        'emb_last_tok_before_gen': tbg_hidden,
        'emb_tok_before_eos':      slt_hidden,
        'logit_feats':             logit_feats,
        'token_ids':               gen_data[0]['token_ids'],
        'num_clusters':            len(clusters),
        'cluster_sizes':           [len(c) for c in clusters],
    }

print("generate_record defined.")

In [ ]:
# Check for existing checkpoint to resume from
import glob

existing_checkpoints = sorted(glob.glob(os.path.join(DATA_DIR, 'checkpoint_*.pkl')))
records = []
start_idx = 0

if existing_checkpoints:
    latest = existing_checkpoints[-1]
    print(f"Found checkpoint: {latest}")
    with open(latest, 'rb') as f:
        records = pickle.load(f)
    start_idx = len(records)
    print(f"Resuming from index {start_idx} ({len(records)} records already collected)")
else:
    print("No checkpoint found. Starting fresh.")
    
print(f"Will process questions {start_idx} to {NUM_QUESTIONS - 1}")

In [ ]:
# Main generation loop
errors = []

for i in tqdm(range(start_idx, NUM_QUESTIONS), desc='Generating records'):
    example = dataset_subset[i]
    
    try:
        record = generate_record(engine, example, num_samples=NUM_SAMPLES)
        records.append(record)
    except Exception as e:
        error_msg = f"[{i}] {example['question'][:60]}: {e}"
        print(f"\nERROR at index {i}: {e}")
        errors.append({'index': i, 'question': example['question'], 'error': str(e)})
        continue
    
    # Save intermediate checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(DATA_DIR, f'checkpoint_{i+1}.pkl')
        with open(ckpt_path, 'wb') as f:
            pickle.dump(records, f)
        print(f"\nCheckpoint saved: {ckpt_path} ({len(records)} records)")

print(f"\nDone. Collected {len(records)} records. Errors: {len(errors)}")

## Section 7 — Dataset Summary Statistics

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for server environments
import matplotlib.pyplot as plt

# Filter to records with hidden states
valid_records = [r for r in records if r['emb_last_tok_before_gen'] is not None]
print(f"Records total:         {len(records)}")
print(f"Records with HS:       {len(valid_records)}")
print(f"Dropped (no HS):       {len(records) - len(valid_records)}")

energy_scores  = np.array([r['energy_score_raw'] for r in valid_records])
entropy_scores = np.array([r['entropy_score_raw'] for r in valid_records])
correctness    = np.array([r['correctness'] for r in valid_records])
num_clusters   = np.array([r['num_clusters'] for r in valid_records])

print(f"\nEnergy scores:")
print(f"  mean={energy_scores.mean():.3f}, std={energy_scores.std():.3f}, min={energy_scores.min():.3f}, max={energy_scores.max():.3f}")
print(f"\nEntropy scores:")
print(f"  mean={entropy_scores.mean():.3f}, std={entropy_scores.std():.3f}, min={entropy_scores.min():.3f}, max={entropy_scores.max():.3f}")
print(f"\nCorrectness:")
print(f"  mean={correctness.mean():.3f}  ({int(correctness.sum())} correct / {len(correctness)} total)")
print(f"\nNum clusters:")
for k in range(1, 7):
    count = (num_clusters == k).sum()
    print(f"  {k} clusters: {count} ({100*count/len(num_clusters):.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(energy_scores, bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('energy_score_raw')
axes[0].set_ylabel('Count')
axes[0].set_title('Energy Score Distribution')
axes[0].axvline(energy_scores.mean(), color='red', linestyle='--', label=f'mean={energy_scores.mean():.3f}')
axes[0].legend()

axes[1].hist(entropy_scores, bins=40, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('entropy_score_raw')
axes[1].set_ylabel('Count')
axes[1].set_title('Entropy Score Distribution')
axes[1].axvline(entropy_scores.mean(), color='red', linestyle='--', label=f'mean={entropy_scores.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plot_path = os.path.join(DATA_DIR, 'teacher_score_distributions.png')
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Plot saved: {plot_path}")

In [ ]:
# Orientation check (quick)
from scipy.stats import spearmanr

rho_e, p_e = spearmanr(energy_scores, correctness)
rho_h, p_h = spearmanr(entropy_scores, correctness)
print(f"Energy  ρ with correctness: {rho_e:+.3f}  (p={p_e:.2e})  [expected: positive]")
print(f"Entropy ρ with correctness: {rho_h:+.3f}  (p={p_h:.2e})  [expected: negative]")

if rho_e <= 0:
    print("WARNING: energy score orientation looks wrong!")
if rho_h >= 0:
    print("WARNING: entropy score orientation looks wrong!")

## Section 8 — Save Final Dataset

In [ ]:
# Use only records with valid hidden states
final_records = valid_records

output_path = os.path.join(DATA_DIR, 'probe_dataset_llama3-8b_triviaqa.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(final_records, f)

size_mb = os.path.getsize(output_path) / 1024**2
print(f"Dataset saved: {output_path}")
print(f"Records: {len(final_records)}")
print(f"File size: {size_mb:.1f} MB")
print()
print("Record fields:")
if final_records:
    for k, v in final_records[0].items():
        if isinstance(v, np.ndarray):
            print(f"  {k}: np.ndarray {v.shape} {v.dtype}")
        elif isinstance(v, dict):
            print(f"  {k}: dict with {len(v)} keys")
        else:
            print(f"  {k}: {type(v).__name__} = {repr(v)[:60]}")

print()
print("Ready for notebook 02_train_se_probes.ipynb")